In [79]:
import pandas as pd
df=pd.read_csv('data.csv')
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


In [62]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())
df['tenure_group'] = pd.qcut(
    df['tenure'],
    q=4,
    labels=['Q1','Q2','Q3','Q4']
)
service_cols = [
    'PhoneService',
    'OnlineSecurity',
    'DeviceProtection',
    'TechSupport',
    'StreamingTV',
    'StreamingMovies'
]

df['service_count'] = (
    df[service_cols]
    .replace({
        'Yes':1,
        'No':0,
        'No internet service':0,
        'No phone service':0
    })
    .sum(axis=1)
)
#df['charge_group'] = pd.qcut(df['MonthlyCharges'], q=4)

x=df.drop(['customerID','Churn'],axis=1)
y = df['Churn'].map({'No':0, 'Yes':1})

C:\Users\SURYANSH\AppData\Local\Temp\ipykernel_27480\1283245500.py:19: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  .replace({


In [60]:
print(df['tenure'].dtype)

category


In [63]:


#seprating column features 
numerical_feat=['tenure','MonthlyCharges','TotalCharges']
categorical_feat=['tenure_group','gender','SeniorCitizen','Partner','Dependents','PhoneService','MultipleLines','InternetService','OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies','Contract','PaperlessBilling','PaymentMethod']

#creating a pipeline
from sklearn import preprocessing
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

preproessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_feat),
        ('cat', OneHotEncoder(), categorical_feat)
    ]
)
transformed_data = preproessor.fit_transform(df)


In [64]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)


In [77]:
from pyexpat import model

from sklearn.linear_model import LogisticRegression
model_pipeline = Pipeline([
    ('preproessor', preproessor),
    ('model', 
     LogisticRegression(
    C=0.5,
    penalty='l2',
    solver='liblinear',
    class_weight='balanced',
    max_iter=1000,
    random_state=42
))
])
model_pipeline.fit(x_train,y_train)
y_pred=model_pipeline.predict(x_test)
y_prob = model_pipeline.predict_proba(x_test)[:,1]

threshold = 0.35

y_pred_custom = (y_prob >= threshold).astype(int)

In [78]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred_custom))     
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test,y_pred_custom))
from sklearn.metrics import accuracy_score
print(accuracy_score(y_test,y_pred_custom))

              precision    recall  f1-score   support

           0       0.96      0.59      0.73      1036
           1       0.45      0.93      0.60       373

    accuracy                           0.68      1409
   macro avg       0.70      0.76      0.67      1409
weighted avg       0.82      0.68      0.70      1409

[[609 427]
 [ 27 346]]
0.6777856635911994
